In [7]:
import json
import requests
from web3 import Web3
from PIL import Image
from io import BytesIO

# Load the ABI from the JSON file
with open("C:/Users/sarat/Mini_Project/PPSA-GAN/ImageLedger/out/ImageLedger.sol/ImageLedger.json") as f:
    contract_data = json.load(f)

abi = contract_data['abi']  # Extract the ABI from the JSON structure

# Connect to the local blockchain
w3 = Web3(Web3.HTTPProvider("http://127.0.0.1:8545"))

# Check if the connection is successful
try:
    w3.eth.block_number  # Triggering a call to check connection
    print("Connected to Anvil!")
except Exception as e:
    print("Failed to connect to Anvil:", str(e))

# Replace with your contract's ABI and address
contract_address = "0xa513E6E4b8f2a923D98304ec87F64353C4D5C853"  # Replace with your contract address
contract_abi = abi

# Instantiate the contract
contract = w3.eth.contract(address=contract_address, abi=contract_abi)

# Define the account and private key (Replace with your values)
account = "0xf39Fd6e51aad88F6F4ce6aB8827279cffFb92266"
private_key = "0xac0974bec39a17e36ba4a6b4d238ff944bacb478cbed5efcae784d7bf4f2ff80"

# Function to fetch an image from IPFS
def fetch_ipfs_image(cid):
    response = requests.post(f"http://127.0.0.1:5001/api/v0/cat", params={'arg': cid})
    if response.status_code == 200:
        image_data = response.content
        img = Image.open(BytesIO(image_data))
        img.show()  # Display the image
    else:
        print("Failed to fetch image from IPFS. Status code:", response.status_code)

# Function to store metadata on the blockchain
def store_metadata(cid, metadata):
    try:
        # Build transaction for storing metadata
        store_txn = contract.functions.storeImageMetadata(cid, metadata).build_transaction({
            'from': account,
            'nonce': w3.eth.get_transaction_count(account),
            'gas': 2000000,
            'gasPrice': w3.to_wei('20', 'gwei')
        })

        # Sign and send the transaction
        signed_store_txn = w3.eth.account.sign_transaction(store_txn, private_key)
        store_txn_hash = w3.eth.send_raw_transaction(signed_store_txn.raw_transaction)
        store_txn_receipt = w3.eth.wait_for_transaction_receipt(store_txn_hash)

        print("Metadata stored successfully!")
        print("Transaction receipt:", store_txn_receipt)
    except Exception as e:
        print("Error storing metadata:", e)

# Function to access an image and log access details
def access_image(cid):
    try:
        # Fetch and display the image from IPFS
        fetch_ipfs_image(cid)

        # Build transaction for accessing the image
        access_txn = contract.functions.accessImage(cid).build_transaction({
            'from': account,
            'nonce': w3.eth.get_transaction_count(account),
            'gas': 2000000,
            'gasPrice': w3.to_wei('20', 'gwei')
        })

        # Sign and send the transaction
        signed_access_txn = w3.eth.account.sign_transaction(access_txn, private_key)
        access_txn_hash = w3.eth.send_raw_transaction(signed_access_txn.raw_transaction)
        access_txn_receipt = w3.eth.wait_for_transaction_receipt(access_txn_hash)

        print("Image accessed successfully!")
        print("Transaction receipt:", access_txn_receipt)
    except Exception as e:
        print("Error accessing image:", e)

# Function to retrieve and display metadata and access history
def display_blockchain_data(cid):
    try:
        # Retrieve metadata
        metadata = contract.functions.imageMetadata(cid).call()
        print(f"Metadata for CID ({cid}): {metadata}")

        # Retrieve access history
        access_history = contract.functions.getAccessHistory(cid).call()
        print(f"Access history for CID ({cid}):")
        for i, record in enumerate(access_history):
            print(f"  Record {i + 1}:")
            print(f"    User: {record[0]}")
            print(f"    Timestamp: {record[1]}")
    except Exception as e:
        print("Error fetching blockchain data:", e)

# Main Execution
cid = "QmQ5D386cY8uX94UNsqSKYwBh53uEQkJqhGGdDpYbkgYBC"  # Replace with your CID
metadata_to_store = "Sample metadata for the image."  # Replace with the metadata you want to store

# Step 1: Store metadata
store_metadata(cid, metadata_to_store)

# Step 2: Access the image and log access details
access_image(cid)

# Step 3: Display the stored data
display_blockchain_data(cid)

Connected to Anvil!
Metadata stored successfully!
Transaction receipt: AttributeDict({'type': 0, 'status': 1, 'cumulativeGasUsed': 49978, 'logs': [AttributeDict({'address': '0xa513E6E4b8f2a923D98304ec87F64353C4D5C853', 'topics': [HexBytes('0xe245b3ed248fd15593e9acaba2300c829263bf35e840dd78512f6045930a3304'), HexBytes('0x000000000000000000000000f39fd6e51aad88f6f4ce6ab8827279cfffb92266')], 'data': HexBytes('0x0000000000000000000000000000000000000000000000000000000000000020000000000000000000000000000000000000000000000000000000000000002e516d51354433383663593875583934554e7371534b5977426835337545516b4a7168474764447059626b67594243000000000000000000000000000000000000'), 'blockHash': HexBytes('0x73989889682a27b3d3fa6342740c24ee45805369e1d5b59669d7890ea5d45e3c'), 'blockNumber': 9, 'blockTimestamp': '0x673ee1b0', 'transactionHash': HexBytes('0xb8e9cafedd5842a365c89a671597b2d40d0459e6e9bb7176709ef21a06880c3a'), 'transactionIndex': 0, 'logIndex': 0, 'removed': False})], 'logsBloom': HexBytes('0x000